# BTW Module 01: Data Ingestion, Schema Validation & Quality Control
**Downstream Bulk Transcriptomics Workbench (`btw`)**

สมุดงานตัวอย่างสาธิตการใช้งาน **FR-1 (Data I/O & Validation)** และ **FR-2 (QC & Normalization)**:
1. การโหลดข้อมูล Count Matrix และ Sample Metadata จากไฟล์ CSV/Excel/Parquet
2. การตรวจจับและแจ้งเตือนข้อผิดพลาดเชิงโครงสร้าง (Missing samples, NaNs, Negative values, Duplicates)
3. การวิเคราะห์สถิติเชิงคุณภาพ (Sample QC & Gene QC)
4. การกรองยีนที่มีการแสดงออกต่ำ (Low-expression gene filtering)
5. การทำ Normalization ผ่าน Thin Helper ของ PyDESeq2 (Median-of-Ratios) และ CPM
6. การส่งออกผลลัพธ์ (Multi-sheet Excel Workbook)

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# นำเข้าโมดูลหลักของ btw
import btw
from btw import set_seed, logger
from btw.io import (
    load_counts,
    load_metadata,
    load_dataset,
    validate_bulk_data,
    export_table,
    export_excel_multisheet,
)
from btw.qc_normalize import (
    compute_sample_qc,
    compute_gene_qc,
    filter_low_expression_genes,
    compute_size_factors,
    normalize_deseq2,
    normalize_cpm,
)

# ตั้งค่า Random Seed จากส่วนกลางเพื่อความแม่นยำและ reproducible
set_seed(42)
print(f"BTW version: {btw.__version__}")

## 1. สร้างชุดข้อมูลตัวอย่าง (Synthetic Bulk RNA-seq Data)
สร้าง Count Matrix (100 genes x 6 samples) และ Metadata สำหรับทดสอบการทำงาน

In [ ]:
genes = [f"GENE_{i:03d}" for i in range(1, 101)]
samples = ["ctrl_1", "ctrl_2", "ctrl_3", "treat_1", "treat_2", "treat_3"]

np.random.seed(42)
base_counts = np.random.negative_binomial(5, 0.01, size=(100, 6))
# กระตุ้นการแสดงออกของ 15 ยีนแรกในกลุ่ม treat
base_counts[:15, 3:] = (base_counts[:15, 3:] * 4.5).astype(int)

counts_df = pd.DataFrame(base_counts, index=genes, columns=samples)
metadata_df = pd.DataFrame({
    "sample_id": samples,
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch2", "batch1", "batch2", "batch1", "batch2"]
}).set_index("sample_id")

print("Counts Matrix:")
display(counts_df.head())
print("\nSample Metadata:")
display(metadata_df)

## 2. การตรวจสอบความสมบูรณ์ของข้อมูล (Schema Validation: FR-1)

In [ ]:
# ตรวจสอบข้อมูลชุดที่ถูกต้อง
report = validate_bulk_data(counts_df, metadata_df)
print(report.summary())

### สาธิตการตรวจจับข้อผิดพลาด (Intentional Error Detection)
ทดสอบว่าระบบจะแจ้งเตือนเมื่อเกิดกรณี sample ID ไม่ตรงกัน หรือมีค่า NaN

In [ ]:
mismatched_meta = metadata_df.copy()
mismatched_meta.index = ["ctrl_1", "ctrl_2", "ctrl_3", "treat_1", "treat_2", "UNKNOWN_SAMPLE"]

err_report = validate_bulk_data(counts_df, mismatched_meta)
print(err_report.summary())

## 3. สถิติเชิงคุณภาพ (Sample & Gene QC: FR-2)

In [ ]:
# 1. Sample QC Summary (Library Size, Detection Rate)
sample_qc = compute_sample_qc(counts_df)
display(sample_qc)

# 2. Gene QC Summary (Mean, Variance, Dispersion, Expression Frequency)
gene_qc = compute_gene_qc(counts_df)
display(gene_qc.head(10))

## 4. การกรองยีนแสดงออกต่ำ (Low-expression Gene Filtering)

In [ ]:
filtered_counts, filter_summary = filter_low_expression_genes(
    counts_df,
    min_counts=10,
    min_samples=3
)
display(filter_summary)
print(f"จำนวนยีนก่อนกรอง: {counts_df.shape[0]} ยีน -> หลังกรอง: {filtered_counts.shape[0]} ยีน")

## 5. Normalization Utilities (FR-2)
เรียกใช้ PyDESeq2 Median-of-Ratios Normalization Helper และ CPM

In [ ]:
# คำนวณ DESeq2 Size Factors
norm_result = normalize_deseq2(filtered_counts, metadata=metadata_df, design_factors="condition")

print("Size Factors:")
print(norm_result.size_factors)

print("\nNormalized Counts (Median-of-Ratios):")
display(norm_result.normalized_counts.head())

print("\nLog2(Normalized Counts + 1):")
display(norm_result.log2_counts.head())

## 6. การส่งออกผลลัพธ์ (Export Tables: FR-1)
ส่งออกข้อมูล QC และ Normalized counts ไปยัง Excel สมุดงานหลายชีต และไฟล์ CSV

In [ ]:
out_dir = Path("results/example_qc")
out_dir.mkdir(parents=True, exist_ok=True)

# ส่งออกเป็น Excel Multi-sheet Workbook
workbook_path = out_dir / "qc_and_normalization_summary.xlsx"
sheets = {
    "Sample_QC": sample_qc,
    "Gene_QC_Top": gene_qc.head(50),
    "Filter_Summary": filter_summary,
    "Normalized_Counts": norm_result.normalized_counts.head(50),
}
saved_excel = export_excel_multisheet(sheets, workbook_path)
print(f"Successfully exported multi-sheet report to: {saved_excel}")